# 02 — System Demo (one problem, end-to-end)

Walk a single problem through all stages: **Stage 0** (role self-assessment) → **Stage 0.5** (deterministic role assignment) → **Stage 1** (independent solving) → **Stage 2** (peer review) → **Stage 3** (refinement) → **Stage 4** (judgment).

This runs with the configured backend. By default `LLM_BACKEND=offline` (deterministic, no API key). Set `LLM_BACKEND=openai` in `.env` to use the real model.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config, utils
from src.stages.stage0_assessment import run_stage0
from src.stages.stage0_5_assignment import assign_roles
from src.stages.stage1_solve import run_stage1
from src.stages.stage2_review import run_stage2
from src.stages.stage3_refine import run_stage3
from src.stages.stage4_judge import run_stage4

logger = utils.setup_logger('notebook_demo')
print('Backend:', config.LLM_BACKEND)

problems = utils.load_json(config.PROBLEMS_PATH)
problem = problems[0]  # prob_001
print('Problem:', problem['problem'])
print('Ground truth:', problem['correct_answer'])

## Stage 0 — Role Self-Assessment

In [ ]:
assessments = run_stage0(problem, logger)
print(json.dumps(assessments, indent=2))

## Stage 0.5 — Deterministic Role Assignment

In [ ]:
assignment = assign_roles(assessments, logger)
assignment

## Stage 1 — Independent Solutions

In [ ]:
solutions = run_stage1(problem, assignment, logger)
for sid, s in solutions.items():
    print(f"{sid} ({s['model']}): {s['final_answer']}  (conf={s['confidence']})")

## Stage 2 — Peer Review (6 reviews)

In [ ]:
reviews = run_stage2(problem, assignment, solutions, logger)
for r in reviews:
    print(f"{r['reviewer_id']} -> {r['solution_reviewed']}: {r['overall_assessment']} ({len(r['evaluation']['errors'])} errors)")

## Stage 3 — Refinement

In [ ]:
refinements = run_stage3(problem, assignment, solutions, reviews, logger)
for sid, r in refinements.items():
    print(f"{sid}: {r['refined_final_answer']}  (conf={r['confidence']})")

## Stage 4 — Final Judgment

In [ ]:
judgment = run_stage4(problem, assignment, solutions, reviews, refinements, logger)
print(json.dumps(judgment, indent=2))
winner = judgment['winner']
final = refinements[winner]['refined_final_answer']
print('\nFINAL ANSWER:', final)
print('GROUND TRUTH:', problem['correct_answer'])
print('CORRECT?', utils.answers_match(final, problem['correct_answer']))